# Laboratorio 2 — Ejercicio 1: Modelos LSTM para Series de Tiempo

**CC3084 – Data Science · Universidad del Valle de Guatemala · Semestre II 2026**

**Integrantes:** Javier España #23361 · Angel Esquit #23221 · Roberto Barreda #23354


## 0. Imports y configuración global

In [ ]:
import os, itertools, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

SEMILLA   = 42
EPOCAS    = 500
VALIDACION = 24          # últimos 24 meses de train para validación interna
TRAIN_RATIO = 0.70
SERIES = ['Total_Consistent', 'Via_Aerea']

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

print(f'PyTorch {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

---
## 1.1  Preparación de datos: conjuntos de entrenamiento y prueba

Se utiliza el mismo CSV del Laboratorio 1 (`series_de_tiempo_completas.csv`). La serie se divide en **70 % entrenamiento** y **30 % prueba**, igual que en el laboratorio anterior, para que las métricas sean comparables.

In [ ]:
# --- Carga y división ---
df = pd.read_csv('data/series_de_tiempo_completas.csv',
                  index_col='Fecha', parse_dates=True).asfreq('MS')

fechas = sorted(df.index.unique())
corte  = pd.Timestamp(fechas[int(len(fechas) * TRAIN_RATIO)])
print(f'Fecha de corte (70 %): {corte:%Y-%m}')

def dividir(col):
    s = df[col].interpolate(method='time').fillna(0).clip(lower=0)
    return s.loc[:corte].asfreq('MS'), s.loc[corte:].iloc[1:].asfreq('MS')

conjuntos = {c: dividir(c) for c in SERIES}

for nombre, (train, test) in conjuntos.items():
    print(f'\n{nombre}:')
    print(f'  Train: {len(train)} obs ({train.index.min():%Y-%m} → {train.index.max():%Y-%m})')
    print(f'  Test : {len(test)} obs  ({test.index.min():%Y-%m} → {test.index.max():%Y-%m})')

In [ ]:
# Visualización de la división train / test
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, nombre in zip(axes, SERIES):
    train, test = conjuntos[nombre]
    ax.plot(train.index, train.values, label='Entrenamiento', color='#2563eb')
    ax.plot(test.index,  test.values,  label='Prueba',        color='#dc2626')
    ax.axvline(corte, color='gray', ls='--', lw=1, label='Corte 70/30')
    ax.set_title(nombre, fontsize=13, fontweight='bold')
    ax.set_ylabel('Visitantes')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('1.1 — División Train / Test', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

---
## 1.2  Definición de modelos LSTM y tuneo de hiperparámetros

Se definen **dos arquitecturas** y una grilla de hiperparámetros. El tuneo evalúa cada combinación sobre los **últimos 24 meses del conjunto de entrenamiento** (validación interna).

| Configuración | Capas LSTM | Dropout | Ventanas | Unidades | Learning Rates |
|---------------|------------|---------|----------|----------|----------------|
| **LstmSimple**  | 1 | 0.0 | 12, 24 | 32, 64 | 0.01, 0.001 |
| **LstmApilado** | 2 | 0.2 | 12, 24 | 32, 64 | 0.01, 0.001 |

Esto genera **8 combinaciones por arquitectura × 2 arquitecturas = 16 modelos** por serie.

In [ ]:
# --- Arquitectura LSTM ---
class RedLstm(nn.Module):
    """Red LSTM configurable: 1 o más capas, con dropout opcional."""
    def __init__(self, unidades, capas, dropout):
        super().__init__()
        self.lstm = nn.LSTM(1, unidades, capas, batch_first=True,
                            dropout=dropout if capas > 1 else 0.0)
        self.salida = nn.Linear(unidades, 1)

    def forward(self, x):
        h, _ = self.lstm(x)
        return self.salida(h[:, -1])

# --- Funciones auxiliares ---
def crear_secuencias(valores, ventana):
    """Genera pares (X, y) con ventanas deslizantes."""
    X = np.array([valores[i:i+ventana] for i in range(len(valores)-ventana)])
    y = np.array([valores[i+ventana]   for i in range(len(valores)-ventana)])
    return (torch.tensor(X, dtype=torch.float32).unsqueeze(-1),
            torch.tensor(y, dtype=torch.float32).unsqueeze(-1))

def entrenar_modelo(x, y, unidades, capas, dropout, lr, epocas=EPOCAS):
    """Entrena una RedLstm y devuelve (modelo, pérdida_final)."""
    torch.manual_seed(SEMILLA)
    modelo = RedLstm(unidades, capas, dropout)
    opt    = torch.optim.Adam(modelo.parameters(), lr=lr)
    crit   = nn.MSELoss()
    modelo.train()
    for _ in range(epocas):
        opt.zero_grad()
        loss = crit(modelo(x), y)
        loss.backward()
        opt.step()
    return modelo, float(loss.detach())

def calc_mae(real, pred):
    return float(np.mean(np.abs(real - pred)))

def calc_rmse(real, pred):
    return float(np.sqrt(np.mean((real - pred)**2)))

print('Arquitectura y funciones definidas ✓')

In [ ]:
# --- Grilla de hiperparámetros ---
CONFIGURACIONES = {
    'LstmSimple':  {'capas': [1], 'dropout': [0.0],
                    'ventana': [12, 24], 'unidades': [32, 64], 'lr': [0.01, 0.001]},
    'LstmApilado': {'capas': [2], 'dropout': [0.2],
                    'ventana': [12, 24], 'unidades': [32, 64], 'lr': [0.01, 0.001]},
}

def gen_combinaciones(rejilla):
    claves = list(rejilla)
    return [dict(zip(claves, v)) for v in itertools.product(*rejilla.values())]

total_combos = sum(len(gen_combinaciones(r)) for r in CONFIGURACIONES.values())
print(f'Total de combinaciones por serie: {total_combos}')

In [ ]:
# --- Ejecución del tuneo ---
def tunear_serie(train):
    """Evalúa todas las combinaciones usando los últimos VALIDACION meses de train."""
    ajuste    = train.iloc[:-VALIDACION]
    escalador = MinMaxScaler().fit(ajuste.values.reshape(-1, 1))
    escalado  = escalador.transform(train.values.reshape(-1, 1)).ravel()
    real_val  = train.values[-VALIDACION:]

    filas = []
    for nombre_cfg, rejilla in CONFIGURACIONES.items():
        for p in gen_combinaciones(rejilla):
            X, Y = crear_secuencias(escalado, p['ventana'])
            corte_val = len(Y) - VALIDACION
            modelo, perdida = entrenar_modelo(
                X[:corte_val], Y[:corte_val],
                p['unidades'], p['capas'], p['dropout'], p['lr'])
            modelo.eval()
            with torch.no_grad():
                pred_esc = modelo(X[corte_val:]).numpy().reshape(-1, 1)
            pred = escalador.inverse_transform(pred_esc).ravel()
            filas.append({
                'Configuracion': nombre_cfg, **p,
                'PerdidaEntrenamiento': round(perdida, 6),
                'ValMae':  round(calc_mae(real_val, pred), 2),
                'ValRmse': round(calc_rmse(real_val, pred), 2),
            })
    return pd.DataFrame(filas)

print('Ejecutando tuneo (16 modelos × 2 series = 32 entrenamientos)…')
print('Esto puede tardar unos minutos.\n')

tablas_tuneo = {}
for nombre in SERIES:
    train, _ = conjuntos[nombre]
    tabla = tunear_serie(train)
    tabla.insert(0, 'Serie', nombre)
    tablas_tuneo[nombre] = tabla
    print(f'✓ Tuneo completado para {nombre}')

tuneo_df = pd.concat(tablas_tuneo.values(), ignore_index=True)
os.makedirs('results', exist_ok=True)
tuneo_df.to_csv('results/TuneoLstm.csv', index=False)
print('\nResultados guardados en results/TuneoLstm.csv')

In [ ]:
# --- Tabla de tuneo para Total_Consistent (ordenada por ValRmse) ---
print('=== Total_Consistent — Resultados de Tuneo ===')
display(tablas_tuneo['Total_Consistent']
        .drop(columns='Serie')
        .sort_values('ValRmse')
        .reset_index(drop=True)
        .style.highlight_min(subset=['ValMae', 'ValRmse'], color='#c6efce'))

In [ ]:
# --- Tabla de tuneo para Via_Aerea (ordenada por ValRmse) ---
print('=== Via_Aerea — Resultados de Tuneo ===')
display(tablas_tuneo['Via_Aerea']
        .drop(columns='Serie')
        .sort_values('ValRmse')
        .reset_index(drop=True)
        .style.highlight_min(subset=['ValMae', 'ValRmse'], color='#c6efce'))

In [ ]:
# --- Selección del mejor modelo global por serie ---
mejor_idx = tuneo_df.groupby('Serie')['ValRmse'].idxmin()
mejores   = tuneo_df.loc[mejor_idx].reset_index(drop=True)
mejores.to_csv('results/MejoresLstm.csv', index=False)

print('═══════════════════════════════════════════════════════════════')
print('  MEJOR MODELO SELECCIONADO POR SERIE (menor ValRmse)')
print('═══════════════════════════════════════════════════════════════')
display(mejores)

---
## 1.3  Predicción sobre el conjunto de prueba con el mejor modelo

Se re-entrena cada mejor modelo usando **todo el conjunto de entrenamiento** (sin reservar validación) y se genera la predicción sobre el conjunto de prueba.

Se emplean dos estrategias de predicción:

- **Multi-step (Autoregresivo):** el modelo se alimenta recursivamente de sus propias predicciones para generar todo el horizonte de prueba. Esta es la predicción verdadera a futuro.
- **One-step ahead:** se utiliza el valor real del paso anterior para predecir el siguiente. Esto da una cota inferior del error y muestra la capacidad del modelo paso a paso.

In [ ]:
def predecir_test(train, test, params, autoregresivo=True):
    """
    Re-entrena el mejor modelo con TODO el train y predice sobre test.
    
    autoregresivo=True  → Multi-step: predicción recursiva (escenario real)
    autoregresivo=False → One-step ahead: usa valores reales de test como contexto
    """
    torch.manual_seed(SEMILLA)
    escalador = MinMaxScaler().fit(train.values.reshape(-1, 1))
    train_esc = escalador.transform(train.values.reshape(-1, 1)).ravel()
    test_esc  = escalador.transform(test.values.reshape(-1, 1)).ravel()

    ventana = params['ventana']
    X_tr, Y_tr = crear_secuencias(train_esc, ventana)
    modelo, _ = entrenar_modelo(X_tr, Y_tr,
                                params['unidades'], params['capas'],
                                params['dropout'],  params['lr'])
    modelo.eval()

    if autoregresivo:
        # --- Predicción recursiva (Multi-step) ---
        historia = list(train_esc[-ventana:])
        preds_esc = []
        for _ in range(len(test)):
            inp = torch.tensor(historia[-ventana:],
                               dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
            with torch.no_grad():
                out = modelo(inp).item()
            preds_esc.append(out)
            historia.append(out)
        preds = escalador.inverse_transform(
            np.array(preds_esc).reshape(-1, 1)).ravel()
    else:
        # --- One-step ahead (usa valores reales como contexto) ---
        full_esc = np.concatenate([train_esc[-ventana:], test_esc])
        X_te = np.array([full_esc[i:i+ventana] for i in range(len(test))])
        inp = torch.tensor(X_te, dtype=torch.float32).unsqueeze(-1)
        with torch.no_grad():
            preds_esc = modelo(inp).numpy().reshape(-1, 1)
        preds = escalador.inverse_transform(preds_esc).ravel()

    return preds

print('Función de predicción definida ✓')

In [ ]:
# --- Ejecutar predicciones con el mejor modelo de cada serie ---
resultados_test = []
predicciones = {}   # almacenar para graficar

for _, row in mejores.iterrows():
    serie = row['Serie']
    train, test = conjuntos[serie]
    params = {
        'unidades': int(row['unidades']),
        'capas':    int(row['capas']),
        'dropout':  float(row['dropout']),
        'ventana':  int(row['ventana']),
        'lr':       float(row['lr']),
    }

    print(f'\nPrediciendo {serie} con: {row["Configuracion"]}, '
          f'ventana={params["ventana"]}, unidades={params["unidades"]}, lr={params["lr"]}…')

    preds_ar = predecir_test(train, test, params, autoregresivo=True)
    preds_os = predecir_test(train, test, params, autoregresivo=False)

    mae_ar  = calc_mae(test.values,  preds_ar)
    rmse_ar = calc_rmse(test.values, preds_ar)
    mae_os  = calc_mae(test.values,  preds_os)
    rmse_os = calc_rmse(test.values, preds_os)

    resultados_test.append({
        'Serie': serie,
        'Configuracion': row['Configuracion'],
        'Ventana': params['ventana'],
        'Unidades': params['unidades'],
        'LR': params['lr'],
        'Test_MAE_Multistep':  round(mae_ar, 2),
        'Test_RMSE_Multistep': round(rmse_ar, 2),
        'Test_MAE_OneStep':    round(mae_os, 2),
        'Test_RMSE_OneStep':   round(rmse_os, 2),
    })
    predicciones[serie] = (preds_ar, preds_os)

    # Guardar predicciones detalladas
    pd.DataFrame({
        'Fecha': test.index,
        'Real': test.values,
        'Pred_Multistep': preds_ar,
        'Pred_OneStep':   preds_os,
    }).to_csv(f'results/predicciones_{serie.lower()}.csv', index=False)

    print(f'  Multi-step →  MAE: {mae_ar:,.2f}  |  RMSE: {rmse_ar:,.2f}')
    print(f'  One-step   →  MAE: {mae_os:,.2f}  |  RMSE: {rmse_os:,.2f}')

df_test = pd.DataFrame(resultados_test)
df_test.to_csv('results/ResultadosTestLSTM.csv', index=False)
print('\n✓ Predicciones guardadas en results/')

In [ ]:
# --- Tabla resumen de resultados en test ---
print('═══════════════════════════════════════════════════════════════════════')
print('  RESULTADOS EN CONJUNTO DE PRUEBA — MEJOR MODELO LSTM POR SERIE')
print('═══════════════════════════════════════════════════════════════════════')
display(df_test.style.format({
    'Test_MAE_Multistep':  '{:,.2f}',
    'Test_RMSE_Multistep': '{:,.2f}',
    'Test_MAE_OneStep':    '{:,.2f}',
    'Test_RMSE_OneStep':   '{:,.2f}',
}))

In [ ]:
# --- Gráficas de predicción ---
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

for ax, serie in zip(axes, SERIES):
    train, test = conjuntos[serie]
    preds_ar, preds_os = predicciones[serie]

    # Mostrar últimos 3 años de train + todo test
    ax.plot(train.index[-36:], train.values[-36:],
            color='#475569', lw=1.5, alpha=0.7, label='Train (últimos 3 años)')
    ax.plot(test.index, test.values,
            color='#2563eb', lw=2, label='Test real')
    ax.plot(test.index, preds_ar,
            color='#dc2626', lw=1.8, ls='--', label='LSTM Multi-step')
    ax.plot(test.index, preds_os,
            color='#16a34a', lw=1.8, ls=':', label='LSTM One-step')
    ax.axvline(test.index[0], color='gray', ls='--', lw=0.8)

    ax.set_title(f'Predicciones LSTM — {serie}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Visitantes')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

fig.suptitle('1.3 — Predicción con el mejor modelo LSTM', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
plt.savefig('results/predicciones_lstm.png', dpi=150, bbox_inches='tight')
plt.show()